# Tech Challenge — Fase 3
## State of Data Brasil 2023–2025

Este notebook reúne a documentação técnica do projeto: arquitetura, decisões de tratamento, exemplos de transformação e consultas usadas nas validações e análises.

> O processamento principal foi executado em **AWS Glue Jobs com PySpark**. Os scripts completos usados no ambiente AWS estão na pasta `scripts/`.

## 1. Objetivo

O trabalho reúne as pesquisas State of Data Brasil de 2023, 2024 e 2025 em um único Data Lake. O principal desafio foi tratar as diferenças entre os questionários sem perder a rastreabilidade dos dados originais.

A análise final foi organizada em sete temas: senioridade, gênero, região, modelo de trabalho, remuneração, tecnologias e Inteligência Artificial.

## 2. Arquitetura utilizada

```text
State of Data / Kaggle
        |
        v
Amazon S3 - Bronze (CSV bruto)
        |
        v
AWS Glue Jobs + PySpark
        |
        v
Amazon S3 - Silver (Parquet)
  - respondentes
  - tecnologias
  - ia
        |
        +--> Glue Crawlers / Data Catalog
        |
        v
AWS Glue Job - Gold Analytics
        |
        v
Amazon S3 - Gold (Parquet)
  - perfil_mercado
  - remuneracao
  - tecnologias
  - ia
        |
        +--> Glue Crawlers / Data Catalog
        |
        v
Amazon Athena
        |
        v
Power BI / apresentação final
```

O diagrama editável está em `arquitetura/Arquitetura_AWS_TechChallenge_Fase3.drawio`.

## 3. Organização do S3

```text
bronze/
  state_of_data/
    ano=2023/
    ano=2024/
    ano=2025/

silver/
  respondentes/
  tecnologias/
  ia/

gold/
  perfil_mercado/
  remuneracao/
  tecnologias/
  ia/

athena-results/
```

A Bronze mantém os CSVs como foram recebidos. Silver e Gold são gravadas em Parquet e particionadas por `ano_pesquisa`.

## 4. Silver de respondentes

Os nomes das perguntas mudam entre os anos. Em vez de tentar unir os três CSVs diretamente, cada pesquisa recebe um mapeamento próprio antes da padronização.

Exemplos:

```text
Senioridade
2023: ('P2_g ', 'Nivel')
2024: 2.g_nivel
2025: 2.g_nivel

Modelo de trabalho atual
2024: 2.r_modelo_de_trabalho_atual
2025: 2.q_modelo_de_trabalho_atual
```

O script completo está em `scripts/tc-fase3-silver-respondentes.py`.

In [ ]:
# Exemplo simplificado do mapeamento usado antes do unionByName
MAP_2024 = {
    "id_respondente": "0.a_token",
    "genero": "1.b_genero",
    "regiao": "1.i.2_regiao_onde_mora",
    "cargo_atual": "2.f_cargo_atual",
    "senioridade": "2.g_nivel",
    "faixa_salarial": "2.h_faixa_salarial",
    "modelo_trabalho_atual": "2.r_modelo_de_trabalho_atual"
}

MAP_2025 = {
    "id_respondente": "0.a_token",
    "genero": "1.b_genero",
    "regiao": "1.i.2_regiao_onde_mora",
    "cargo_atual": "2.f_cargo_atual",
    "senioridade": "2.g_nivel",
    "faixa_salarial": "2.h_faixa_salarial",
    "modelo_trabalho_atual": "2.q_modelo_de_trabalho_atual"
}

### Validação

A deduplicação foi feita por `ano_pesquisa + id_respondente`.

| Ano | Respondentes |
|---:|---:|
| 2023 | 5.293 |
| 2024 | 5.215 |
| 2025 | 3.494 |
| **Total** | **14.002** |

Como o tamanho da amostra muda entre os anos, as comparações finais usam principalmente percentuais.

## 5. Silver de tecnologias

Nos CSVs, as tecnologias aparecem em várias colunas binárias. Para facilitar consultas e rankings, essas colunas foram transformadas em linhas.

```text
Origem
id | SQL | Python | Power BI
A  |  1  |   1    |    0

Silver
id | categoria | tecnologia
A  | linguagem | SQL
A  | linguagem | Python
```

As categorias tratadas são linguagem, banco de dados, cloud e BI.

Script: `scripts/tc-fase3-silver-tecnologias.py`.

In [ ]:
# Ideia central da transformação wide -> long
item = F.when(
    F.col("4.d.3_Python").cast("double") == 1.0,
    F.struct(
        F.lit("linguagem").alias("categoria"),
        F.lit("Python").alias("tecnologia")
    )
)

# No script completo, a mesma lógica é aplicada dinamicamente
# às colunas identificadas pelos prefixos de cada ano.

### Comparabilidade

Duas diferenças do questionário foram mantidas de forma explícita:

- em 2025, a pergunta de linguagens mede **preferência**;
- a pergunta de Cloud de 2023 tem uma formulação diferente das edições seguintes.

Por isso, a Silver guarda uma coluna de comparabilidade que também é levada para a Gold.

## 6. Silver de Inteligência Artificial

As perguntas de IA foram organizadas em dimensões padronizadas:

- `prioridade_empresa`
- `tipo_uso_empresa`
- `produtividade_pessoal`
- `barreira_adocao`
- `resultado_llm_empresa`

A última dimensão existe somente em 2025, portanto não é apresentada como série histórica.

Script: `scripts/tc-fase3-silver-ia.py`.

In [ ]:
# Exemplo simplificado da padronização de prioridade de IA
texto = F.lower(F.trim(F.col("3.e_ai_generativa_e_llm_é_uma_prioridade?")))

prioridade_padronizada = (
    F.when(texto.contains("não é uma iniciativa"), "nao_prioridade")
     .when(texto.contains("principal prioridade"), "prioridade_principal")
     .when(texto.contains("principais prioridades"), "alta_prioridade_2_4_anos")
     .when(texto.contains("uma das várias iniciativas"), "iniciativa_secundaria")
     .when(texto.contains("não sei"), "nao_sabe")
)

A Silver de IA resultou em **40.363 respostas/seleções**. Esse total é maior que o número de respondentes porque algumas perguntas permitem múltiplas escolhas.

## 7. Camada Gold

Um único Glue Job lê as três Silvers e produz quatro tabelas agregadas:

| Tabela | Uso principal |
|---|---|
| `gold_perfil_mercado` | perfil, senioridade, gênero, região e modelo de trabalho |
| `gold_remuneracao` | distribuição salarial por diferentes dimensões |
| `gold_tecnologias` | adoção, ranking e comparabilidade das tecnologias |
| `gold_ia` | indicadores das perguntas de IA |

Script: `scripts/tc-fase3-gold-analytics.py`.

In [ ]:
# Exemplo do cálculo de percentual usado na Gold
contagem = (
    base.groupBy("ano_pesquisa", "senioridade")
        .agg(F.countDistinct("id_respondente").alias("quantidade"))
)

totais = (
    base.groupBy("ano_pesquisa")
        .agg(F.countDistinct("id_respondente").alias("total_validos"))
)

gold = (
    contagem.join(totais, "ano_pesquisa")
            .withColumn(
                "percentual",
                F.round(F.col("quantidade") * 100.0 / F.col("total_validos"), 2)
            )
)

### Regras adotadas na Gold

- Cada percentual usa o conjunto de respostas válidas da própria dimensão.
- Em perguntas de múltipla seleção, a soma dos percentuais pode ultrapassar 100%.
- A faixa salarial original foi preservada; registros anômalos foram sinalizados em vez de alterados silenciosamente.
- `Especialista/Staff+`, presente em 2025, foi mantido como categoria própria.

## 8. Catálogo e Athena

Os Crawlers registraram as tabelas no database `tc_fase3` e reconheceram `ano_pesquisa` como partição.

Tabelas Silver:

```text
silver_respondentes
silver_tecnologias
silver_ia
```

Tabelas Gold:

```text
gold_perfil_mercado
gold_remuneracao
gold_tecnologias
gold_ia
```

O Athena foi usado tanto para conferir as transformações quanto para gerar os resultados enviados ao Power BI.

## 9. Exemplos de consultas no Athena

O arquivo `sql/consultas_athena.sql` contém a lista completa. Abaixo ficam alguns exemplos representativos.

### Validação da quantidade de respondentes

```sql
SELECT
    ano_pesquisa,
    COUNT(*) AS quantidade_respondentes
FROM silver_respondentes
GROUP BY ano_pesquisa
ORDER BY ano_pesquisa;
```

### Validação dos volumes da Gold

```sql
SELECT 'perfil_mercado' AS tabela, COUNT(*) AS linhas
FROM gold_perfil_mercado
UNION ALL
SELECT 'remuneracao', COUNT(*) FROM gold_remuneracao
UNION ALL
SELECT 'tecnologias', COUNT(*) FROM gold_tecnologias
UNION ALL
SELECT 'ia', COUNT(*) FROM gold_ia;
```

### Senioridade

```sql
SELECT
    ano_pesquisa,
    categoria AS senioridade,
    quantidade,
    total_validos,
    percentual,
    ranking
FROM gold_perfil_mercado
WHERE dimensao = 'senioridade'
ORDER BY ano_pesquisa, ranking;
```

### Top tecnologias

```sql
SELECT
    ano_pesquisa,
    categoria,
    ranking,
    tecnologia,
    qtd_profissionais,
    percentual_adocao,
    comparabilidade
FROM gold_tecnologias
WHERE ranking <= 5
ORDER BY ano_pesquisa, categoria, ranking;
```

### IA

```sql
SELECT
    ano_pesquisa,
    dimensao,
    resposta,
    qtd_respondentes,
    percentual,
    ranking,
    percentuais_somam_100
FROM gold_ia
WHERE dimensao IN ('prioridade_empresa', 'resultado_llm_empresa')
ORDER BY ano_pesquisa, dimensao, ranking;
```

## 10. Validações finais

| Tabela Gold | Linhas |
|---|---:|
| `gold_perfil_mercado` | 186 |
| `gold_remuneracao` | 964 |
| `gold_tecnologias` | 207 |
| `gold_ia` | 86 |

Esses números correspondem às tabelas agregadas usadas nas análises finais.

## 11. Principais resultados

- Em 2025, Sênior + Especialista/Staff+ representam **48,28%** dos respondentes com senioridade informada.
- A participação feminina passa de **24,43% em 2023 para 21,95% em 2025**.
- O Sudeste representa **64,41%** dos respondentes em 2025.
- O trabalho 100% remoto passa de **46,31% em 2023 para 39,70% em 2025**.
- A soma de prioridade principal e alta prioridade para IA passa de **36,16% em 2023 para 60,58% em 2025**.
- Em 2025, **38,39%** dos respondentes da pergunta sobre resultados com LLMs relatam pilotos com pouco impacto; **26,47%** relatam produção com impacto.

## 12. Cuidados de interpretação

- As amostras não têm o mesmo tamanho em todos os anos.
- `Especialista/Staff+` aparece somente em 2025.
- A pergunta de linguagens de 2025 mede preferência.
- A pergunta de Cloud de 2023 tem uma formulação diferente.
- A pergunta sobre resultados com LLMs existe somente em 2025.
- Os resultados descrevem os respondentes da pesquisa e não devem ser tratados como uma estimativa censitária do mercado brasileiro.

## 13. Arquivos da entrega

```text
TechChallenge_Fase3/
├── scripts/
│   ├── tc-fase3-silver-respondentes.py
│   ├── tc-fase3-silver-tecnologias.py
│   ├── tc-fase3-silver-ia.py
│   └── tc-fase3-gold-analytics.py
├── sql/
│   └── consultas_athena.sql
├── notebook/
│   └── TechChallenge_Fase3.ipynb
├── arquitetura/
│   └── Arquitetura_AWS_TechChallenge_Fase3.drawio
├── powerbi/
├── apresentacao/
└── README.md
```